In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [35]:
# Read UN Comtrade Data (Total Commodity Codes)

comtrade_folder = Path("data/comtrade_data")
comtrade_files = []
for file_path in comtrade_folder.iterdir():
    if file_path.is_file():
        print(f"Opening {file_path.name}")
        comtrade_data_file = pd.read_csv(file_path, index_col=False)
        comtrade_files.append(comtrade_data_file)

total_comtrade_data = pd.concat(comtrade_files)
total_comtrade_data = total_comtrade_data[total_comtrade_data['partnerISO'] != 'World']



Opening TradeData_03_06.csv
Opening TradeData_07_10.csv
Opening TradeData_11_14.csv
Opening TradeData_15_18.csv
Opening TradeData_19_22.csv
Opening TradeData_98_02.csv


In [77]:
# Simplify Comtrade data to essentials.
comtrade_data = total_comtrade_data[['refYear', 'reporterISO', 'partnerISO', 'fobvalue']]
comtrade_data["refYear"] = comtrade_data["refYear"].astype(int)
print(comtrade_data.head())

   refYear reporterISO partnerISO      fobvalue
0     2003         AUS        ALB  7.181132e+06
1     2003         AUS        W00  7.028824e+10
2     2003         AUS        AFG  2.410862e+06
3     2003         AUS        ATA  7.486340e+05
4     2003         AUS        DZA  3.413292e+07


In [79]:
# Read WDI GDP Data, convert to long format
wdi_gdp_data = pd.read_csv('data/wdi_gdp.csv', skiprows=4)
wdi_gdp_data = wdi_gdp_data.loc[:, ~wdi_gdp_data.columns.str.startswith('Unnamed')]
wdi_gdp_data = wdi_gdp_data.drop(columns=["Indicator Name", "Indicator Code"])
wdi_gdp_long = wdi_gdp_data.melt(id_vars=["Country Name", "Country Code"], var_name='year', value_name='gdp')
wdi_gdp_long = wdi_gdp_long[wdi_gdp_long['year'].astype(int) > 1997]
wdi_gdp_long["year"] = wdi_gdp_long["year"].astype(int)
print(wdi_gdp_long.head())

                      Country Name Country Code  year           gdp
10108                        Aruba          ABW  1998  1.665363e+09
10109  Africa Eastern and Southern          AFE  1998  2.687286e+11
10110                  Afghanistan          AFG  1998           NaN
10111   Africa Western and Central          AFW  1998  2.971265e+11
10112                       Angola          AGO  1998  6.506222e+09


In [83]:
# Combine Comtrade and GDP data.
full_trade_data = pd.merge(comtrade_data, wdi_gdp_long,
                           left_on=["refYear", "reporterISO"],
                           right_on=["year", "Country Code"],
                           how="left").drop(columns=['Country Name', 'Country Code', 'year'])
full_trade_data = full_trade_data.rename(columns={"gdp": "reporterGDP"})
full_trade_data = pd.merge(full_trade_data, wdi_gdp_long,
                           left_on=["refYear", "partnerISO"],
                           right_on=["year", "Country Code"],
                           how="left").drop(columns=['Country Name', 'Country Code', 'year'])
full_trade_data = full_trade_data.rename(columns={"gdp": "partnerGDP"})

print(full_trade_data.head())

   refYear reporterISO partnerISO      fobvalue   reporterGDP    partnerGDP
0     2003         AUS        ALB  7.181132e+06  4.685172e+11  5.801712e+09
1     2003         AUS        W00  7.028824e+10  4.685172e+11           NaN
2     2003         AUS        AFG  2.410862e+06  4.685172e+11  4.520947e+09
3     2003         AUS        ATA  7.486340e+05  4.685172e+11           NaN
4     2003         AUS        DZA  3.413292e+07  4.685172e+11  7.348226e+10


In [84]:
full_trade_data.to_csv('processed data/comtrade_gdp.csv', index=False)